In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

# Transforming exogenous features
> Compute transformations on your exogenous features for MLForecast

The MLForecast class allows you to compute lag transformations on your target, however, sometimes you want to also compute transformations on your dynamic exogenous features. This guide shows you how to accomplish that.

## Data setup

In [ ]:
from mlforecast.utils import generate_series, generate_prices_for_series

In [ ]:
series = generate_series(10, equal_ends=True)
prices = generate_prices_for_series(series)
prices.head(2)

Suppose that you have some series along with their prices for each id and date and you want to compute forecasts for the next 7 days. Since the price is a dynamic feature you have to provide the future values through `X_df` in `MLForecast.predict`.

If you want to use not only the price but the lag7 of the price and the expanding mean of the lag1 for example, you can compute them before training, merge them with your series and then provide the future values through `X_df`. Consider the following example.

## Computing the transformations

In [ ]:
from window_ops.expanding import expanding_mean

from mlforecast.feature_engineering import transform_exog

In [ ]:
transformed_prices = transform_exog(prices, lags=[7], lag_transforms={1: [expanding_mean]})
transformed_prices.head(10)

You can now merge this with your original series

In [ ]:
series_with_prices = series.merge(transformed_prices, on=['unique_id', 'ds'])
series_with_prices.head(10)

You can then define your forecast object. Note that you can still compute lag features based on the target as you normally would.

In [ ]:
from sklearn.linear_model import LinearRegression

from mlforecast import MLForecast

In [ ]:
fcst = MLForecast(
    models=[LinearRegression()],
    freq='D',
    lags=[1],
    date_features=['dayofweek'],
)
fcst.preprocess(series_with_prices, dropna=True).head()

It's important to note that the `dropna` argument only considers the null values generated by the lag features based on the target. If you want to drop all rows containing null values you have to do that in your original series.

In [ ]:
series_with_prices2 = series_with_prices.dropna()
fcst.preprocess(series_with_prices2, dropna=True, static_features=[]).head()

You can now train the model.

In [ ]:
fcst.fit(series_with_prices2, static_features=[])

And predict using the prices. Note that you can provide the dataframe with the full history and mlforecast will filter the required dates for the forecasting horizon.

In [ ]:
fcst.predict(1, X_df=transformed_prices).head()

In this example we have prices for the next 7 days, if you try to forecast a longer horizon you'll get an error.

In [ ]:
from fastcore.test import test_fail

In [ ]:
test_fail(lambda: fcst.predict(8, X_df=transformed_prices), contains='Found missing inputs in X_df')